In [ ]:
pip install zarr

In [ ]:
pip install dask-ml

In [ ]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr
import os

import cv2
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import torch
import dask.array as da
from dask_ml.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import preprocess_input
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
import shutil

In [ ]:
print(torch.__version__)

use_gpu = torch.cuda.is_available()

print(use_gpu)

2.3.1+cu121
True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Reading file from GoogleDrive
data = zarr.open("/content/drive/My Drive/Team Project X-Rays/Dataframes/data_2.1.zarr", mode='r')
target = zarr.open("/content/drive/My Drive/Team Project X-Rays/Dataframes/target_2.1.zarr", mode='r')

In [ ]:
# Check TensorFlow version and GPU availability
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

TensorFlow version: 2.17.0
Num GPUs Available:  1


In [ ]:
#Train test split
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 42, shuffle = True)

In [ ]:
#Checking shape
print(X_train.shape)
print(y_train.shape)

(16885, 89401)
(16885,)


In [ ]:
#Initializing the scaler
scaler = StandardScaler()

#Fitting on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

#Transforming the test data using the same scaler
X_test_scaled = scaler.transform(X_test)

In [ ]:
#SVM
svm = SVC(gamma = 0.01, kernel = "poly")

In [ ]:
import time
start_time = time.time()

#Fitting model
svm.fit(X_train_scaled, y_train)

# Measure time
model1_time = (time.time() - start_time)/60

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

SVC(gamma=0.01, kernel='poly')

In [ ]:
from sklearn import datasets
import joblib

In [ ]:
from sklearn import datasets
import joblib

#Saving model
joblib.dump(svm, '/content/drive/My Drive/svm_model.joblib')

['/content/drive/My Drive/Team Project X-Rays/svm_model.joblib']

In [ ]:
if os.path.exists('/content/drive/My Drive/Team Project X-Rays/svm_model.joblib'):
    print(f"Model successfully saved at '/content/drive/My Drive/Team Project X-Rays/svm_model.joblib'")
else:
    print("Failed to save the model.")

Model successfully saved at '/content/drive/My Drive/Team Project X-Rays/svm_model.joblib'


In [ ]:
svm = joblib.load('/content/drive/My Drive/svm_model.joblib')

In [ ]:
#Predicting on the test set
y_pred = svm.predict(X_test)

#Accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy - SVM: {test_accuracy:.4f}")

#F1 - Score
f1_score = f1_score(y_test, y_pred, average = "macro")
print(f"Test F1-score (weighted) - SVM: {f1_score:.4f}")

#Classification Report
print("Classification Report - SVM:")
print(classification_report(y_test, y_pred))

#Confusion Matrix
print("Confusion Matrix - SVM:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy - SVM: 0.3229
Test F1-score (weighted) - SVM: 0.3122
Classification Report - SVM:
              precision    recall  f1-score   support

         0.0       0.48      0.29      0.36      2028
         1.0       0.19      0.20      0.20       688
         2.0       0.30      0.34      0.32      1249
         3.0       0.24      0.79      0.37       256

    accuracy                           0.32      4221
   macro avg       0.30      0.41      0.31      4221
weighted avg       0.37      0.32      0.32      4221

Confusion Matrix - SVM:
[[595 336 749 348]
 [223 139 226 100]
 [388 237 427 197]
 [ 39   6   9 202]]


In [ ]:
#KNN
knn = KNeighborsClassifier(n_neighbors = 7, metric = 'minkowski')

In [ ]:
import time
start_time = time.time()

#Fitting model
knn.fit(X_train_scaled, y_train)

# Measure time
model1_time = (time.time() - start_time)/60

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

Model 1: --- 0.009312585989634196 minutes ---


In [ ]:
#Predicting on the test set
y_pred = knn.predict(X_test)

#Accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy - KNN: {test_accuracy:.4f}")

#F1 - Score
f1_score = f1_score(y_test, y_pred, average = "macro")
print(f"Test F1-score (weighted) - KNN: {f1_score:.4f}")

#Classification Report
print("Classification Report - KNN:")
print(classification_report(y_test, y_pred))

#Confusion Matrix
print("Confusion Matrix - KNN:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy - KNN: 0.2691
Test F1-score (weighted) - KNN: 0.1994
Classification Report - KNN:
              precision    recall  f1-score   support

           0       0.17      0.85      0.29       689
           1       0.69      0.19      0.30      2068
           2       0.54      0.13      0.21      1203
           3       0.00      0.00      0.00       262

    accuracy                           0.27      4222
   macro avg       0.35      0.29      0.20      4222
weighted avg       0.52      0.27      0.25      4222

Confusion Matrix - KNN:
[[ 588   62   39    0]
 [1593  390   85    0]
 [ 935  110  158    0]
 [ 246    3   13    0]]


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
#LR
lr = LogisticRegression(max_iter = 10000, solver = 'lbfgs', C = 1.0)

In [ ]:
import time
start_time = time.time()

#Fitting model
lr.fit(X_train_scaled, y_train)

# Measure time
model1_time = (time.time() - start_time)/60

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

LogisticRegression(max_iter=10000)

In [ ]:
from sklearn import datasets
import joblib

#Saving model
joblib.dump(lr, '/content/drive/My Drive/lr_model.joblib')

['/content/drive/My Drive/lr_model.joblib']

In [ ]:
#Predicting on the test set
y_pred = lr.predict(X_test)

#Accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy - LR: {test_accuracy:.4f}")

#F1 - Score
f1_score = f1_score(y_test, y_pred, average = "macro")
print(f"Test F1-score (weighted) - LR: {f1_score:.4f}")

#Classification Report
print("Classification Report - LR:")
print(classification_report(y_test, y_pred))

#Confusion Matrix
print("Confusion Matrix - LR:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy - LR: 0.6055
Classification Report - LR:
              precision    recall  f1-score   support

         0.0       0.66      0.82      0.73      2028
         1.0       0.31      0.34      0.32       688
         2.0       0.69      0.39      0.50      1249
         3.0       0.72      0.69      0.70       256

    accuracy                           0.61      4221
   macro avg       0.59      0.56      0.56      4221
weighted avg       0.62      0.61      0.59      4221

Confusion Matrix - LR:
[[1658  218  120   32]
 [ 349  233   97    9]
 [ 438  294  489   28]
 [  57   18    5  176]]


In [ ]:
#DT
dt = DecisionTreeClassifier(criterion = "entropy", max_depth = 4, random_state = 123)

In [ ]:
import time
start_time = time.time()

#Fitting model
dt.fit(X_train_scaled, y_train)

# Measure time
model1_time = (time.time() - start_time)/60

# Modelling Time
print("Model 1: --- %s minutes ---" % model1_time)

Model 1: --- 6.685418113072713 minutes ---


In [ ]:
#Predicting on the test set
y_pred = dt.predict(X_test)

#Accuracy
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy - DT: {test_accuracy:.4f}")

#F1 - Score
#f1_score = f1_score(y_test, y_pred, average = "macro")
#print(f"Test F1-score (weighted) - DT: {f1_score:.4f}")

#Classification Report
print("Classification Report - DT:")
print(classification_report(y_test, y_pred))

#Confusion Matrix
print("Confusion Matrix - DT:")
print(confusion_matrix(y_test, y_pred))

Test Accuracy - DT: 0.4697
Classification Report - DT:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       689
           1       0.49      0.93      0.64      2068
           2       0.21      0.05      0.08      1203
           3       0.00      0.00      0.00       262

    accuracy                           0.47      4222
   macro avg       0.18      0.24      0.18      4222
weighted avg       0.30      0.47      0.34      4222

Confusion Matrix - DT:
[[   0  616   73    0]
 [   0 1924  144    0]
 [   0 1144   59    0]
 [   0  262    0    0]]


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
